# 03 — Tokenization: Words, Sentences & Subwords

**Learning objective.** Compare naïve whitespace, regex/NLTK word tokenization, and a simple subword-style decomposition.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


## Why tokenization matters
Models operate on units. Tokenization defines those units and therefore the vocabulary, sequence length, OOV behavior, memory footprint and often the model’s error surface.

In [2]:
from nltk.tokenize import TreebankWordTokenizer, sent_tokenize
text = "Don't split $10.50 incorrectly; Akilan's model is state-of-the-art."
whitespace = text.split()
treebank = TreebankWordTokenizer().tokenize(text)
print('Whitespace:', whitespace)
print('Treebank  :', treebank)

Whitespace: ["Don't", 'split', '$10.50', 'incorrectly;', "Akilan's", 'model', 'is', 'state-of-the-art.']
Treebank  : ['Do', "n't", 'split', '$', '10.50', 'incorrectly', ';', 'Akilan', "'s", 'model', 'is', 'state-of-the-art', '.']


In [3]:
# Educational WordPiece-like greedy splitter with an explicit toy vocabulary.
vocab={'natural','language','process','##ing','token','##ization','deep','learn','##ing','<UNK>'}
def greedy_subword(word):
    word=word.lower(); out=[]; start=0
    while start < len(word):
        found=None
        for end in range(len(word), start, -1):
            piece=word[start:end] if start==0 else '##'+word[start:end]
            if piece in vocab:
                found=(piece,end); break
        if not found: return ['<UNK>']
        out.append(found[0]); start=found[1]
    return out
for w in ['processing','learning','unknownword']:
    print(w, '->', greedy_subword(w))

processing -> ['process', '##ing']
learning -> ['learn', '##ing']
unknownword -> ['<UNK>']


### Modern tokenizer note
Production transformer tokenizers use learned vocabularies and algorithms such as WordPiece, BPE or Unigram. The greedy function above exists to make the mechanics inspectable; it is **not** a replacement for a trained Hugging Face tokenizer.

---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Explain why tokenization changes model cost
- Describe why subwords reduce OOV failures